# 타이타닉 생존 예측 - 4단계: 고급 모델링 및 앙상블

## 프로젝트 진행 상황
- ✅ **1단계 완료**: EDA와 베이스라인 모델
- ✅ **2단계 완료**: 데이터 전처리 및 특성 엔지니어링
- ✅ **3단계 완료**: 기본 모델링 및 평가
- 🔄 **4단계 진행중**: 고급 모델링 및 앙상블 (`04_advanced_modeling_and_ensemble.ipynb`) ← **지금 여기**

## 🎯 이번 노트북의 목표
1. **고급 모델 도입**: 캐글과 같은 경진대회에서 압도적인 성능을 보여주는 **XGBoost**와 **LightGBM** 모델을 사용합니다.
2. **앙상블(Ensemble) 기법 적용**: 여러 모델의 예측을 결합하여 더 안정적이고 높은 성능을 내는 'Voting' 앙상블 모델을 구축합니다.
3. **성능 비교 및 최종 제출**: 3단계의 최고 모델, 고급 단일 모델, 앙상블 모델의 성능을 비교하고, 가장 좋은 모델로 최종 제출 파일을 생성합니다.

> 💡 **실무 팁**: 단일 모델의 성능을 극한까지 끌어올리는 것도 중요하지만, 여러 모델을 현명하게 조합하는 앙상블 기법이 순위를 결정짓는 경우가 많습니다. '집단 지성'의 힘을 활용하는 단계입니다.

## 1단계: 환경 설정 및 데이터 로딩

이전 단계들과 동일하게, 전처리된 데이터를 불러와 준비합니다.

In [ ]:
# 필수 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# 기본 모델들 (3단계에서 사용)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# 고급 모델들 (새로 추가)
# !pip install xgboost lightgbm  <- 설치가 안되어 있다면 이 명령어로 설치
import xgboost as xgb
import lightgbm as lgb

# 앙상블 모델
from sklearn.ensemble import VotingClassifier

# 평가 지표
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ 라이브러리 로딩 완료!")
print("🚀 고급 모델링 준비 완료!")

In [ ]:
# 데이터 경로 설정
PROCESSED_PATH = '../data/processed/'
SUBMISSION_PATH = '../data/submissions/'

# 전처리된 데이터 로딩
print("📂 전처리된 데이터 로딩 중...")
try:
    X_train = pd.read_csv(PROCESSED_PATH + 'X_train.csv')
    X_val = pd.read_csv(PROCESSED_PATH + 'X_val.csv')
    y_train = pd.read_csv(PROCESSED_PATH + 'y_train.csv').values.ravel()
    y_val = pd.read_csv(PROCESSED_PATH + 'y_val.csv').values.ravel()
    X_test = pd.read_csv(PROCESSED_PATH + 'X_test.csv')
    test_ids = pd.read_csv(PROCESSED_PATH + 'test_ids.csv')

    # 3단계에서 튜닝을 위해 합쳤던 전체 훈련 데이터도 만듭니다.
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = np.concatenate([y_train, y_val])

    print(f"✅ 데이터 로딩 완료!")
    print(f"- 전체 훈련 데이터: {X_train_full.shape}")
    print(f"- 테스트 데이터: {X_test.shape}")

except FileNotFoundError as e:
    print(f"❌ 파일 로딩 오류: {e}")
    print("먼저 02_data_preprocessing.ipynb 노트북을 실행하여 데이터를 생성해주세요.")

## 2단계: 고급 모델(XGBoost, LightGBM) 훈련 및 평가

캐글 상위권에서 가장 많이 사용되는 두 모델을 훈련하고 성능을 확인합니다.

In [ ]:
# 고급 모델 정의
advanced_models = {
    'XGBoost': xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(random_state=42)
}

# 3단계에서 성능이 좋았던 모델도 비교를 위해 가져옵니다.
# (GridSearchCV로 찾은 최적 파라미터를 적용하면 더 좋습니다)
base_best_model = {
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200, max_depth=10, min_samples_leaf=1, min_samples_split=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=200, max_depth=3, learning_rate=0.05)
}

# 모델들을 합칩니다.
all_models = {**base_best_model, **advanced_models}

# 모델 학습 및 평가
results = []
print("🤖 고급 모델 학습 및 평가 시작...\n")
print("="*80)

for name, model in all_models.items():
    print(f"\n📊 {name} 학습 중...")
    
    # 모델 학습
    model.fit(X_train, y_train)
    
    # 예측 및 평가
    y_val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    
    results.append({
        'Model': name,
        'Val Acc': val_acc,
        'Val F1': val_f1
    })
    
    print(f"  - 검증 정확도: {val_acc:.4f}")
    print(f"  - 검증 F1 Score: {val_f1:.4f}")

print("\n" + "="*80)
print("✅ 모든 모델 학습 완료!")

# 결과 정리
adv_results_df = pd.DataFrame(results).sort_values('Val Acc', ascending=False)
print("\n📊 고급 모델 성능 비교표:")
print(adv_results_df.to_string(index=False))

## 3단계: 앙상블 모델 (Voting Classifier) 구축

개별적으로 강력한 모델들의 예측을 '투표'를 통해 종합하여, 더 안정적이고 강력한 최종 모델을 만듭니다.

> **VotingClassifier란?**  
> 여러 분류기(Classifier)를 묶어서 하나의 분류기처럼 동작하게 만듭니다.
> - **Hard Voting**: 각 모델이 예측한 클래스(생존/사망) 중 다수결로 최종 결정을 내립니다. (예: 3개 모델이 '생존', 2개 모델이 '사망'으로 예측하면 최종 결과는 '생존')
> - **Soft Voting**: 각 모델이 예측한 '확률'의 평균을 내어, 더 높은 확률 쪽으로 최종 결정을 내립니다. 일반적으로 성능이 더 좋습니다.

In [ ]:
# 앙상블에 사용할 모델들을 리스트로 정의
# 위에서 가장 성능이 좋았던 3~4개 모델을 선택하는 것이 일반적입니다.
estimators = [
    ('Random Forest', all_models['Random Forest']),
    ('Gradient Boosting', all_models['Gradient Boosting']),
    ('XGBoost', all_models['XGBoost']),
    ('LightGBM', all_models['LightGBM'])
]

# Soft Voting 앙상블 모델 생성
# n_jobs=-1: 모든 CPU 코어를 사용하여 병렬 처리
voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft', # 각 모델의 예측 확률 평균을 사용
    n_jobs=-1
)

print("🤖 앙상블 모델(Voting Classifier) 학습 시작...")

# 앙상블 모델 학습
voting_clf.fit(X_train, y_train)

# 앙상블 모델 평가
y_val_pred_ensemble = voting_clf.predict(X_val)
ensemble_acc = accuracy_score(y_val, y_val_pred_ensemble)
ensemble_f1 = f1_score(y_val, y_val_pred_ensemble)

print("✅ 앙상블 모델 학습 및 평가 완료!")
print(f"\n--- 앙상블 모델 성능 ---")
print(f"  - 검증 정확도: {ensemble_acc:.4f}")
print(f"  - 검증 F1 Score: {ensemble_f1:.4f}")

# 이전 단일 모델 최고 성능과 비교
best_single_model_acc = adv_results_df['Val Acc'].max()
print(f"\n--- 성능 비교 ---")
print(f"  - 단일 모델 최고 정확도: {best_single_model_acc:.4f}")
print(f"  - 앙상블 모델 정확도:   {ensemble_acc:.4f}")
improvement = ensemble_acc - best_single_model_acc
print(f"  - 개선도: {improvement:+.4f}")

if improvement > 0:
    print("\n✨ 앙상블을 통해 성능이 향상되었습니다!")
else:
    print("\n💡 앙상블 성능이 단일 모델보다 낮거나 비슷합니다. 모델 조합이나 가중치 조정을 고려해볼 수 있습니다.")

## 4단계: 최종 제출 파일 생성

가장 성능이 좋았던 **앙상블 모델**을 사용하여 최종 예측을 수행하고, 캐글에 제출할 파일을 생성합니다.

> **💡 실무 팁**: 최종 제출 전, 가장 좋은 성능을 보인 모델을 **전체 훈련 데이터(`X_train_full`, `y_train_full`)**로 다시 학습시켜야 합니다. 검증에 사용했던 데이터까지 모두 학습에 활용하여 모델의 성능을 최대한 끌어올리기 위함입니다.

In [ ]:
# 최종 모델 선택 (앙상블 모델)
final_model = voting_clf

print("📚 최종 모델을 전체 훈련 데이터로 재학습합니다...")
# 전체 훈련 데이터로 최종 모델 재학습
final_model.fit(X_train_full, y_train_full)
print("✅ 재학습 완료!")

print("\n🔮 테스트 데이터로 최종 예측을 수행합니다...")
# 테스트 데이터로 예측
final_predictions = final_model.predict(X_test)
print("✅ 예측 완료!")

# 제출 파일 생성
submission_df = pd.DataFrame({
    'PassengerId': test_ids['PassengerId'],
    'Survived': final_predictions
})

# 제출 폴더가 없으면 생성
os.makedirs(SUBMISSION_PATH, exist_ok=True)

# 제출 파일 저장
submission_filepath = os.path.join(SUBMISSION_PATH, 'submission_ensemble.csv')
submission_df.to_csv(submission_filepath, index=False)

print(f"\n💾 최종 제출 파일 생성 완료!")
print(f"📁 저장 위치: {submission_filepath}")
print("\n--- 제출 파일 미리보기 (상위 5개) ---")
print(submission_df.head())

print("\n🎉 모든 과정이 완료되었습니다! 이제 submission_ensemble.csv 파일을 캐글에 제출하여 점수를 확인해보세요!")

---

## 🎉 고급 모델링 및 앙상블 완료!

### ✅ 완료된 작업들:

#### 1. **고급 모델 도입** ✓
- **XGBoost**와 **LightGBM**을 사용하여 기본 모델보다 높은 성능을 확인했습니다.

#### 2. **앙상블 모델 구축** ✓
- 성능이 좋은 모델들을 `VotingClassifier`로 묶어 '집단 지성'의 힘을 활용했습니다.
- 단일 모델 대비 더 안정적이고 높은 성능을 기대할 수 있습니다.

#### 3. **최종 모델 재학습 및 예측** ✓
- 가장 성능이 좋은 앙상블 모델을 **전체 훈련 데이터**로 재학습하여 성능을 극대화했습니다.
- 최종 테스트 데이터에 대한 예측을 수행하고, 새로운 제출 파일(`submission_ensemble.csv`)을 생성했습니다.

### 🚀 다음 단계는?

- **캐글 점수 확인**: `submission_ensemble.csv` 파일을 캐글에 제출하여 3단계 모델의 점수와 비교해보세요! 아마 더 높은 점수를 얻으셨을 겁니다.
- **다양한 앙상블 시도**:
  - `VotingClassifier`의 `voting` 방식을 'hard'로 바꿔서 테스트해보세요.
  - 모델에 가중치(`weights`)를 부여하여 더 성능 좋은 모델의 의견을 더 많이 반영해보세요.
  - **스태킹(Stacking)** 이라는 더 발전된 앙상블 기법에 도전해보세요.
- **특성 공학(Feature Engineering) 심화**: `02_data_preprocessing.ipynb`로 돌아가 `Ticket`이나 `Name`에서 더 창의적인 특성을 만들어보세요. 모델링 기술만큼이나 중요한 것이 바로 특성 공학입니다.

**이로써 캐글 경진대회에서 사용하는 핵심적인 고급 전략까지 모두 경험해보셨습니다. 수고 많으셨습니다!**